# Week 12: CNN Visualization
# 第 12 週：CNN 視覺化 -- 卷積核、特徵圖、CAM/Grad-CAM

## 學習目標 Learning Objectives
1. 理解卷積運算 (Convolution) 的數學原理，從 1D 到 2D
2. 應用常見卷積核（邊緣偵測、模糊、銳化）並視覺化效果
3. 掌握 Padding 和 Stride 對輸出尺寸的影響
4. 實作 Max Pooling 和 Average Pooling
5. 視覺化典型 CNN 架構
6. 觀察多個濾波器產生的特徵圖
7. 理解 CNN 的特徵提取 vs 傳統方法
8. 掌握 CAM/Grad-CAM 的概念與簡化實作
9. 視覺化感受野 (Receptive Field)

In [ ]:
# 匯入必要套件 Import required packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.gridspec import GridSpec

from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from scipy.signal import convolve2d
from scipy.ndimage import convolve

import warnings
warnings.filterwarnings('ignore')

# 設定中文字型
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

np.random.seed(42)

# 載入 digits 資料集
digits = load_digits()
X_digits = digits.data
y_digits = digits.target
images = digits.images  # 8x8 images

print(f'Digits dataset: {X_digits.shape[0]} samples, image size = {images[0].shape}')
print(f'Classes: {np.unique(y_digits)}')

# Show sample digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(images[i], cmap='gray', interpolation='nearest')
    ax.set_title(f'Label: {y_digits[i]}', fontsize=11)
    ax.axis('off')
plt.suptitle('Sample Digits from sklearn (8x8 pixels)', fontsize=14)
plt.tight_layout()
plt.show()

print('All packages imported successfully!')

---
## Part 1: 什麼是卷積 What is Convolution

卷積運算是 CNN 的核心。本質上，一個小窗口（卷積核）在訊號/影像上滑動，每到一個位置就做加權求和。

### 2D 卷積公式

$$(I * K)(i, j) = \sum_{m=0}^{k_h-1} \sum_{n=0}^{k_w-1} I(i+m, j+n) \cdot K(m, n)$$

In [ ]:
# Part 1a: 1D Convolution from scratch
def conv1d(signal, kernel):
    """1D convolution (cross-correlation) from scratch."""
    n = len(signal)
    k = len(kernel)
    output_size = n - k + 1
    output = np.zeros(output_size)
    for i in range(output_size):
        output[i] = np.sum(signal[i:i+k] * kernel)
    return output

# Create a sample signal
t = np.linspace(0, 4*np.pi, 200)
signal = np.sin(t) + 0.5 * np.sin(3*t) + 0.3 * np.random.randn(len(t))

# Different 1D kernels
smooth_kernel = np.ones(15) / 15  # moving average
edge_kernel = np.array([-1, 0, 1])  # edge detection

smoothed = conv1d(signal, smooth_kernel)
edges = conv1d(signal, edge_kernel)

fig, axes = plt.subplots(3, 1, figsize=(14, 8))

axes[0].plot(t, signal, 'b-', linewidth=1, alpha=0.8)
axes[0].set_title('Original Signal', fontsize=13)
axes[0].set_ylabel('Amplitude', fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t[:len(smoothed)], smoothed, 'g-', linewidth=2)
axes[1].set_title('After Smoothing Kernel (Moving Average, k=15)', fontsize=13)
axes[1].set_ylabel('Amplitude', fontsize=11)
axes[1].grid(True, alpha=0.3)

axes[2].plot(t[:len(edges)], edges, 'r-', linewidth=1)
axes[2].set_title('After Edge Detection Kernel [-1, 0, 1]', fontsize=13)
axes[2].set_ylabel('Amplitude', fontsize=11)
axes[2].set_xlabel('Time', fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.suptitle('1D Convolution Examples', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

print('1D convolution: slide a kernel along the signal')
print('Smoothing kernel: averages nearby values -> reduces noise')
print('Edge kernel: highlights changes -> detects transitions')

In [ ]:
# Part 1b: 2D Convolution from scratch
def conv2d(image, kernel):
    """2D convolution (cross-correlation) from scratch."""
    ih, iw = image.shape
    kh, kw = kernel.shape
    oh = ih - kh + 1
    ow = iw - kw + 1
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            output[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return output

# Use a digit image
test_img = images[0].astype(float)  # digit '0'

# Simple edge detection kernel
kernel_edge = np.array([[-1, -1, -1],
                        [ 0,  0,  0],
                        [ 1,  1,  1]])

result = conv2d(test_img, kernel_edge)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(test_img, cmap='gray')
axes[0].set_title(f'Input Image (8x8)\nDigit: {y_digits[0]}', fontsize=12)
axes[0].axis('off')

axes[1].imshow(kernel_edge, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title('Kernel (3x3)\nHorizontal Edge', fontsize=12)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{kernel_edge[i,j]:d}', ha='center', va='center',
                     fontsize=14, fontweight='bold',
                     color='white' if abs(kernel_edge[i,j]) > 0 else 'black')
axes[1].axis('off')

axes[2].imshow(result, cmap='gray')
axes[2].set_title(f'Output Feature Map ({result.shape[0]}x{result.shape[1]})', fontsize=12)
axes[2].axis('off')

plt.suptitle('2D Convolution: Input * Kernel = Feature Map', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Input size: {test_img.shape}')
print(f'Kernel size: {kernel_edge.shape}')
print(f'Output size: {result.shape}  (= {test_img.shape[0]}-{kernel_edge.shape[0]}+1 = {test_img.shape[0]-kernel_edge.shape[0]+1})')

---
## Part 2: 常見卷積核 Common Kernels

不同的卷積核偵測不同的影像特徵：

| 卷積核 | 效果 |
|--------|------|
| Sobel X | 垂直邊緣偵測 |
| Sobel Y | 水平邊緣偵測 |
| Laplacian | 全方向邊緣偵測 |
| Gaussian Blur | 平滑化 |
| Sharpen | 增強細節 |

In [ ]:
# 常見卷積核及其效果

# Generate a larger synthetic image for better visualization
np.random.seed(42)
synth_img = np.zeros((32, 32))
# Add some shapes
synth_img[5:15, 10:20] = 1.0  # rectangle
synth_img[18:28, 5:15] = 0.7  # another rectangle
# Add a circle
for i in range(32):
    for j in range(32):
        if (i-22)**2 + (j-24)**2 < 25:
            synth_img[i, j] = 0.9
# Add some noise
synth_img += np.random.randn(32, 32) * 0.05
synth_img = np.clip(synth_img, 0, 1)

kernels = {
    'Sobel X\n(Vertical edges)': np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    'Sobel Y\n(Horizontal edges)': np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
    'Laplacian\n(All edges)': np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]]),
    'Gaussian Blur\n(Smoothing)': np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16.0,
    'Sharpen\n(Enhance details)': np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Show original image
axes[0, 0].imshow(synth_img, cmap='gray')
axes[0, 0].set_title('Original Image', fontsize=12)
axes[0, 0].axis('off')

for idx, (name, kernel) in enumerate(kernels.items()):
    row = (idx + 1) // 3
    col = (idx + 1) % 3
    ax = axes[row, col]
    
    result = convolve2d(synth_img, kernel, mode='same', boundary='symm')
    ax.imshow(result, cmap='gray')
    ax.set_title(name, fontsize=11)
    ax.axis('off')

plt.suptitle('Common Convolution Kernels and Their Effects', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

print('Edge detection kernels highlight boundaries between regions')
print('Gaussian blur smooths the image by averaging nearby pixels')
print('Sharpen kernel enhances fine details by boosting the center pixel')

In [ ]:
# 在 digits 上應用卷積核
fig, axes = plt.subplots(4, 6, figsize=(18, 12))

digit_indices = [0, 1, 11, 7]  # digits 0, 1, 2, 7
kernel_list = [
    ('Original', None),
    ('Sobel X', np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])),
    ('Sobel Y', np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])),
    ('Laplacian', np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])),
    ('Blur', np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16.0),
    ('Sharpen', np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])),
]

for row, d_idx in enumerate(digit_indices):
    img = images[d_idx].astype(float)
    for col, (k_name, kernel) in enumerate(kernel_list):
        ax = axes[row, col]
        if kernel is None:
            ax.imshow(img, cmap='gray')
        else:
            result = convolve2d(img, kernel, mode='same', boundary='symm')
            ax.imshow(result, cmap='gray')
        if row == 0:
            ax.set_title(k_name, fontsize=11, fontweight='bold')
        if col == 0:
            ax.set_ylabel(f'Digit {y_digits[d_idx]}', fontsize=11, fontweight='bold')
        ax.axis('off')

plt.suptitle('Convolution Kernels Applied to Digit Images', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

print('Different kernels extract different features from the same image')
print('In CNNs, these kernels are LEARNED automatically through backpropagation!')

---
## Part 3: 填充和步幅 Padding and Stride

填充 (Padding) 和步幅 (Stride) 控制輸出特徵圖的尺寸：

$$O = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1$$

- $H$: 輸入尺寸, $p$: padding, $k$: kernel size, $s$: stride

In [ ]:
# Padding and Stride visualization
def conv2d_with_params(image, kernel, padding=0, stride=1):
    """2D convolution with padding and stride."""
    if padding > 0:
        image = np.pad(image, padding, mode='constant', constant_values=0)
    ih, iw = image.shape
    kh, kw = kernel.shape
    oh = (ih - kh) // stride + 1
    ow = (iw - kw) // stride + 1
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            si, sj = i * stride, j * stride
            output[i, j] = np.sum(image[si:si+kh, sj:sj+kw] * kernel)
    return output

test_img_5 = np.arange(1, 26).reshape(5, 5).astype(float)
kernel_3 = np.ones((3, 3)) / 9.0  # averaging kernel

configs = [
    ('Valid (p=0, s=1)', 0, 1),
    ('Same (p=1, s=1)', 1, 1),
    ('Stride=2 (p=0, s=2)', 0, 2),
    ('Stride=2 (p=1, s=2)', 1, 2),
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

# Show input
ax0 = axes[0]
ax0.imshow(test_img_5, cmap='Blues', vmin=0, vmax=25)
for i in range(5):
    for j in range(5):
        ax0.text(j, i, f'{test_img_5[i,j]:.0f}', ha='center', va='center',
                 fontsize=10, fontweight='bold')
ax0.set_title(f'Input (5x5)', fontsize=11)
ax0.axis('off')

for idx, (name, pad, stride) in enumerate(configs):
    result = conv2d_with_params(test_img_5, kernel_3, padding=pad, stride=stride)
    ax = axes[idx + 1]
    ax.imshow(result, cmap='Blues', vmin=0, vmax=25)
    for i in range(result.shape[0]):
        for j in range(result.shape[1]):
            ax.text(j, i, f'{result[i,j]:.1f}', ha='center', va='center',
                    fontsize=9, fontweight='bold')
    h_in = 5
    h_out = (h_in + 2*pad - 3) // stride + 1
    ax.set_title(f'{name}\nOutput: {result.shape[0]}x{result.shape[1]}\n'
                 f'Formula: ({h_in}+2*{pad}-3)/{stride}+1={h_out}', fontsize=10)
    ax.axis('off')

plt.suptitle('Padding & Stride Effects on Output Size (3x3 kernel)', fontsize=14)
plt.tight_layout()
plt.show()

print('Output size formula: O = floor((H + 2p - k) / s) + 1')
print('Valid (no padding): output shrinks')
print('Same padding: output = input size (when s=1)')
print('Stride > 1: downsamples the output')

---
## Part 4: 池化操作 Pooling Operations

池化層進行降維 (Downsampling)，減少計算量並增強魯棒性。

- **Max Pooling**: 取窗口中最大值 -- 保留最強特徵
- **Average Pooling**: 取窗口中平均值 -- 保留整體資訊

In [ ]:
# 池化操作 from scratch
def max_pool2d(image, pool_size=2, stride=2):
    """Max pooling from scratch."""
    ih, iw = image.shape
    oh = (ih - pool_size) // stride + 1
    ow = (iw - pool_size) // stride + 1
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            si, sj = i * stride, j * stride
            output[i, j] = np.max(image[si:si+pool_size, sj:sj+pool_size])
    return output

def avg_pool2d(image, pool_size=2, stride=2):
    """Average pooling from scratch."""
    ih, iw = image.shape
    oh = (ih - pool_size) // stride + 1
    ow = (iw - pool_size) // stride + 1
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            si, sj = i * stride, j * stride
            output[i, j] = np.mean(image[si:si+pool_size, sj:sj+pool_size])
    return output

# Test on a 4x4 input
pool_input = np.array([
    [1, 3, 2, 1],
    [5, 2, 1, 3],
    [4, 8, 6, 2],
    [3, 1, 4, 5]
], dtype=float)

max_result = max_pool2d(pool_input, pool_size=2, stride=2)
avg_result = avg_pool2d(pool_input, pool_size=2, stride=2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Input
ax0 = axes[0]
ax0.imshow(pool_input, cmap='YlOrRd', vmin=0, vmax=8)
for i in range(4):
    for j in range(4):
        ax0.text(j, i, f'{pool_input[i,j]:.0f}', ha='center', va='center',
                 fontsize=16, fontweight='bold')
# Draw pool regions
for si in [0, 2]:
    for sj in [0, 2]:
        rect = mpatches.Rectangle((sj-0.5, si-0.5), 2, 2, linewidth=3,
                                  edgecolor='blue', facecolor='none', linestyle='--')
        ax0.add_patch(rect)
ax0.set_title('Input (4x4)', fontsize=12)
ax0.axis('off')

# Max Pooling
ax1 = axes[1]
ax1.imshow(max_result, cmap='YlOrRd', vmin=0, vmax=8)
for i in range(2):
    for j in range(2):
        ax1.text(j, i, f'{max_result[i,j]:.0f}', ha='center', va='center',
                 fontsize=20, fontweight='bold')
ax1.set_title('Max Pool (2x2, stride=2)\nKeeps strongest activations', fontsize=12)
ax1.axis('off')

# Average Pooling
ax2 = axes[2]
ax2.imshow(avg_result, cmap='YlOrRd', vmin=0, vmax=8)
for i in range(2):
    for j in range(2):
        ax2.text(j, i, f'{avg_result[i,j]:.1f}', ha='center', va='center',
                 fontsize=18, fontweight='bold')
ax2.set_title('Avg Pool (2x2, stride=2)\nKeeps average features', fontsize=12)
ax2.axis('off')

plt.suptitle('Pooling Operations: Max vs Average', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Input: {pool_input.shape} -> Pooled: {max_result.shape}')
print(f'Max Pool result: {max_result}')
print(f'Avg Pool result: {avg_result}')

In [ ]:
# 在 digit 圖像上比較 Max Pooling 和 Average Pooling
fig, axes = plt.subplots(3, 5, figsize=(16, 9))

for col, d_idx in enumerate([0, 1, 11, 7, 15]):
    img = images[d_idx].astype(float)
    max_p = max_pool2d(img, pool_size=2, stride=2)
    avg_p = avg_pool2d(img, pool_size=2, stride=2)

    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title(f'Original (8x8)\nDigit {y_digits[d_idx]}', fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].imshow(max_p, cmap='gray')
    axes[1, col].set_title(f'Max Pool (4x4)', fontsize=10)
    axes[1, col].axis('off')

    axes[2, col].imshow(avg_p, cmap='gray')
    axes[2, col].set_title(f'Avg Pool (4x4)', fontsize=10)
    axes[2, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Max Pool', fontsize=11, fontweight='bold')
axes[2, 0].set_ylabel('Avg Pool', fontsize=11, fontweight='bold')

plt.suptitle('Max Pooling vs Average Pooling on Digit Images', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('Max pooling: preserves sharp features and strong activations')
print('Avg pooling: preserves overall structure but softer')
print('Both reduce spatial dimensions by half (2x2 pool, stride=2)')

---
## Part 5: CNN 架構圖 CNN Architecture Diagram

典型 CNN 架構: Input -> Conv -> Pool -> Conv -> Pool -> Flatten -> FC -> Output

每一層改變特徵圖的空間尺寸和通道數。

In [ ]:
# 視覺化典型 CNN 架構
fig, ax = plt.subplots(figsize=(18, 6))
ax.axis('off')

# Layer configurations: (name, width, height, depth/channels, x_pos)
layers = [
    ('Input\n8x8x1', 8, 8, 1, 0.5),
    ('Conv1\n6x6x4', 6, 6, 4, 2.5),
    ('Pool1\n3x3x4', 3, 3, 4, 4.5),
    ('Conv2\n1x1x8', 1, 1, 8, 6.5),
    ('Flatten\n8', 1, 8, 1, 8.5),
    ('FC\n16', 1, 16, 1, 10.5),
    ('Output\n10', 1, 10, 1, 12.5),
]

colors = ['#E3F2FD', '#BBDEFB', '#90CAF9', '#64B5F6', '#42A5F5', '#2196F3', '#1976D2']

for idx, (name, w, h, d, x) in enumerate(layers):
    # Draw each layer as a rectangle
    rect_h = h * 0.3
    rect_w = max(w * 0.3, 0.5)
    depth_w = min(d * 0.15, 1.0)

    # Main rectangle
    rect = mpatches.FancyBboxPatch((x - rect_w/2, 2 - rect_h/2), rect_w, rect_h,
                                    boxstyle="round,pad=0.1",
                                    facecolor=colors[idx], edgecolor='black',
                                    linewidth=1.5)
    ax.add_patch(rect)

    # Depth indicator (3D effect)
    if d > 1:
        for dd in range(min(d, 3)):
            offset = dd * 0.08
            rect_d = mpatches.FancyBboxPatch(
                (x - rect_w/2 + offset, 2 - rect_h/2 + offset),
                rect_w, rect_h, boxstyle="round,pad=0.1",
                facecolor=colors[idx], edgecolor='black',
                linewidth=0.8, alpha=0.3)
            ax.add_patch(rect_d)

    # Label
    ax.text(x, 2, name, ha='center', va='center', fontsize=9, fontweight='bold')

    # Arrow to next layer
    if idx < len(layers) - 1:
        next_x = layers[idx + 1][4]
        ax.annotate('', xy=(next_x - max(layers[idx+1][1]*0.3, 0.5)/2 - 0.1, 2),
                    xytext=(x + rect_w/2 + 0.1, 2),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Operation labels
ops = ['Conv 3x3', 'MaxPool\n2x2', 'Conv 3x3', 'Flatten', 'Dense', 'Softmax']
op_x = [1.5, 3.5, 5.5, 7.5, 9.5, 11.5]
for op, x in zip(ops, op_x):
    ax.text(x, 3.5, op, ha='center', va='center', fontsize=8,
            style='italic', color='#666666',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.set_xlim(-0.5, 14)
ax.set_ylim(0, 4.5)
ax.set_title('Typical CNN Architecture: Feature Extractor + Classifier', fontsize=14, pad=20)

# Legend
ax.text(0, 0.3, 'Feature Extractor (Conv + Pool)', fontsize=10, color='blue',
        fontweight='bold')
ax.text(8, 0.3, 'Classifier (FC layers)', fontsize=10, color='blue',
        fontweight='bold')

plt.tight_layout()
plt.show()

print('CNN Architecture:')
print('  Feature Extractor: Conv layers extract spatial features, Pool layers downsample')
print('  Classifier: Flatten features into 1D, Dense layers make predictions')
print('  Spatial dimensions decrease while channel count increases through the network')

---
## Part 6: 特徵圖視覺化 Feature Map Visualization

當我們將多個不同的卷積核應用到同一張圖像上，每個核產生一張特徵圖。
這些特徵圖共同構成該層的輸出，代表了不同空間模式的檢測結果。

In [ ]:
# 模擬 CNN 第一層的多濾波器特徵圖
np.random.seed(42)

# Define a set of learned-like filters
filters = {
    'Horizontal': np.array([[-1, -1, -1], [2, 2, 2], [-1, -1, -1]]),
    'Vertical': np.array([[-1, 2, -1], [-1, 2, -1], [-1, 2, -1]]),
    'Diagonal /': np.array([[2, -1, -1], [-1, 2, -1], [-1, -1, 2]]),
    'Diagonal \\': np.array([[-1, -1, 2], [-1, 2, -1], [2, -1, -1]]),
    'Center': np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]]),
    'Blur': np.ones((3, 3)) / 9.0,
    'Random 1': np.random.randn(3, 3) * 0.5,
    'Random 2': np.random.randn(3, 3) * 0.5,
}

# Apply all filters to several digit images
digit_samples = [0, 1, 11, 7]  # digits 0, 1, 2, 7

for d_idx in digit_samples:
    img = images[d_idx].astype(float)
    fig, axes = plt.subplots(1, 9, figsize=(20, 2.5))

    axes[0].imshow(img, cmap='gray')
    axes[0].set_title(f'Input\n(digit {y_digits[d_idx]})', fontsize=9)
    axes[0].axis('off')

    for f_idx, (f_name, f_kernel) in enumerate(filters.items()):
        fmap = convolve2d(img, f_kernel, mode='same', boundary='symm')
        fmap = np.maximum(fmap, 0)  # ReLU activation
        axes[f_idx + 1].imshow(fmap, cmap='hot')
        axes[f_idx + 1].set_title(f_name, fontsize=8)
        axes[f_idx + 1].axis('off')

    plt.suptitle(f'Feature Maps for Digit {y_digits[d_idx]} (8 filters + ReLU)',
                 fontsize=12)
    plt.tight_layout()
    plt.show()

print('Each filter detects a different pattern in the input image')
print('Horizontal filter: responds to horizontal edges')
print('Vertical filter: responds to vertical strokes')
print('Center filter: responds to bright spots')
print('In a real CNN, these filters are learned automatically!')

---
## Part 7: CNN approach vs Flat MLP on Digit Recognition

CNN 利用空間結構（2D 卷積），而 MLP 將像素展平 (flatten) 為 1D 向量。
我們用 sklearn 的 `MLPClassifier` 來演示扁平化方法，同時展示手動特徵提取如何改善效能。

In [ ]:
# 方法 1: 直接用展平的像素值
X_flat = X_digits  # already flattened (1797, 64)

# 方法 2: 手動卷積特徵提取 (simulating what Conv layers do)
def extract_conv_features(images_arr, kernels_dict):
    """Extract features using manual convolution (simulating CNN conv layers)."""
    features_list = []
    for img in images_arr:
        img_features = []
        for k_name, kernel in kernels_dict.items():
            fmap = convolve2d(img, kernel, mode='same', boundary='symm')
            fmap = np.maximum(fmap, 0)  # ReLU
            # Pool: global average and max
            img_features.append(fmap.mean())
            img_features.append(fmap.max())
            img_features.append(fmap.std())
            # Spatial pooling: 2x2 regions
            pooled = max_pool2d(fmap, pool_size=2, stride=2)
            img_features.extend(pooled.ravel())
        features_list.append(img_features)
    return np.array(features_list)

conv_kernels = {
    'H-edge': np.array([[-1, -1, -1], [2, 2, 2], [-1, -1, -1]]),
    'V-edge': np.array([[-1, 2, -1], [-1, 2, -1], [-1, 2, -1]]),
    'Diag1': np.array([[2, -1, -1], [-1, 2, -1], [-1, -1, 2]]),
    'Diag2': np.array([[-1, -1, 2], [-1, 2, -1], [2, -1, -1]]),
    'Center': np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]]),
    'Blur': np.ones((3, 3)) / 9.0,
}

X_conv = extract_conv_features(images, conv_kernels)

print(f'Flat pixel features shape: {X_flat.shape}')
print(f'Conv-extracted features shape: {X_conv.shape}')

# Compare performance
results = {}
for name, X_data in [('Flat Pixels', X_flat), ('Conv Features', X_conv)]:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_data)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_scaled, y_digits, test_size=0.3, random_state=42, stratify=y_digits)

    mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500,
                        random_state=42, early_stopping=True)
    mlp.fit(X_tr, y_tr)
    train_acc = mlp.score(X_tr, y_tr)
    test_acc = mlp.score(X_te, y_te)
    results[name] = (train_acc, test_acc)
    print(f'{name:20s}: Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}')

# Plot comparison
fig, ax = plt.subplots(figsize=(8, 5))
names = list(results.keys())
train_accs = [results[n][0] for n in names]
test_accs = [results[n][1] for n in names]

x_pos = np.arange(len(names))
width = 0.3
bars1 = ax.bar(x_pos - width/2, train_accs, width, label='Train', color='#2196F3', alpha=0.8)
bars2 = ax.bar(x_pos + width/2, test_accs, width, label='Test', color='#FF5722', alpha=0.8)

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=11)

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Flat Pixels vs Conv-Extracted Features', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(names, fontsize=12)
ax.legend(fontsize=11)
ax.set_ylim([0.8, 1.02])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\nConv features capture spatial patterns (edges, corners) that raw pixels miss')
print('A real CNN does this automatically, learning optimal filters via backprop')

---
## Part 8: CAM 概念 -- 類別激活映射 Class Activation Mapping

CAM 找出 CNN 做決策時最關注的影像區域：

$$M_c(x, y) = \sum_k w_k^c \cdot f_k(x, y)$$

其中 $w_k^c$ 是分類層對類別 $c$ 的權重，$f_k$ 是最後卷積層的第 $k$ 個特徵圖。

我們用一個簡化版本來演示核心概念。

In [ ]:
# 簡化版 CAM/Grad-CAM 概念演示
np.random.seed(42)

def simplified_cam(image, kernels_list, class_weights):
    """
    Simplified CAM demonstration.
    Applies kernels to image, then uses class weights to create activation map.
    """
    feature_maps = []
    for kernel in kernels_list:
        fmap = convolve2d(image, kernel, mode='same', boundary='symm')
        fmap = np.maximum(fmap, 0)  # ReLU
        feature_maps.append(fmap)

    # Weighted combination (simulating CAM)
    cam = np.zeros_like(image)
    for k, fmap in enumerate(feature_maps):
        cam += class_weights[k] * fmap

    # Normalize to [0, 1]
    cam = np.maximum(cam, 0)
    if cam.max() > 0:
        cam = cam / cam.max()
    return cam, feature_maps

# Define kernels and class-specific weights
cam_kernels = [
    np.array([[-1, -1, -1], [2, 2, 2], [-1, -1, -1]]),  # horizontal
    np.array([[-1, 2, -1], [-1, 2, -1], [-1, 2, -1]]),  # vertical
    np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]]),  # center
    np.array([[2, -1, -1], [-1, 2, -1], [-1, -1, 2]]),  # diagonal
]

# Different class weights for different digits
# (Simulating what the FC layer learns)
class_weight_sets = {
    'Digit 0 focus\n(curves)': [0.5, 0.5, 0.2, 0.3],
    'Digit 1 focus\n(vertical)': [0.1, 1.0, 0.5, 0.1],
    'Digit 7 focus\n(horizontal+diagonal)': [0.8, 0.2, 0.3, 0.7],
}

test_digits = [0, 1, 7]  # digits 0, 1, 7
test_indices = [0, 1, 7]

fig, axes = plt.subplots(len(test_indices), 4, figsize=(16, 10))

for row, (d_idx, (cw_name, cw)) in enumerate(
        zip(test_indices, class_weight_sets.items())):
    img = images[d_idx].astype(float)

    # Original
    axes[row, 0].imshow(img, cmap='gray')
    axes[row, 0].set_title(f'Original\nDigit {y_digits[d_idx]}', fontsize=11)
    axes[row, 0].axis('off')

    # CAM overlay
    cam, fmaps = simplified_cam(img, cam_kernels, cw)

    axes[row, 1].imshow(cam, cmap='jet')
    axes[row, 1].set_title(f'CAM Heatmap\n{cw_name}', fontsize=10)
    axes[row, 1].axis('off')

    # Overlay
    axes[row, 2].imshow(img, cmap='gray', alpha=0.5)
    axes[row, 2].imshow(cam, cmap='jet', alpha=0.5)
    axes[row, 2].set_title('Overlay\n(Image + CAM)', fontsize=10)
    axes[row, 2].axis('off')

    # Feature maps
    fmap_combined = np.hstack([f / (f.max() + 1e-8) for f in fmaps])
    axes[row, 3].imshow(fmap_combined, cmap='hot')
    axes[row, 3].set_title('Feature Maps\n(4 filters)', fontsize=10)
    axes[row, 3].axis('off')

plt.suptitle('Simplified CAM: Which regions does the model focus on?', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('CAM Interpretation:')
print('- Hot (red/yellow) regions = high contribution to the predicted class')
print('- Cool (blue) regions = low contribution')
print('- Different class weights highlight different spatial patterns')
print('- Grad-CAM extends this to use gradients instead of FC weights')

---
## Part 9: 感受野視覺化 Receptive Field Visualization

感受野 (Receptive Field) 是特徵圖上一個元素「看到」的原始輸入區域大小。
隨著層數加深，感受野逐漸增大：
- Layer 1 (3x3 conv): RF = 3x3
- Layer 2 (3x3 conv): RF = 5x5
- Layer 3 (3x3 conv): RF = 7x7

In [ ]:
# 感受野視覺化
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Input image
input_size = 9
colors_rf = plt.cm.Set3(np.linspace(0, 1, 12))

# Layer 0: Input
ax = axes[0]
ax.set_xlim(-0.5, input_size - 0.5)
ax.set_ylim(-0.5, input_size - 0.5)
ax.set_aspect('equal')
for i in range(input_size):
    for j in range(input_size):
        rect = mpatches.Rectangle((j-0.5, i-0.5), 1, 1,
                                  facecolor='lightblue', edgecolor='gray', linewidth=0.5)
        ax.add_patch(rect)
ax.set_title(f'Input (9x9)', fontsize=12)
ax.invert_yaxis()
ax.set_xticks([])
ax.set_yticks([])

# Layer 1: 3x3 conv -> output 7x7, RF=3x3
ax = axes[1]
out1_size = 7
ax.set_xlim(-0.5, out1_size - 0.5)
ax.set_ylim(-0.5, out1_size - 0.5)
ax.set_aspect('equal')
for i in range(out1_size):
    for j in range(out1_size):
        rect = mpatches.Rectangle((j-0.5, i-0.5), 1, 1,
                                  facecolor='lightyellow', edgecolor='gray', linewidth=0.5)
        ax.add_patch(rect)
# Highlight center pixel's RF
rect = mpatches.Rectangle((3-0.5, 3-0.5), 1, 1,
                          facecolor='red', edgecolor='black', linewidth=2, alpha=0.7)
ax.add_patch(rect)
ax.set_title(f'After Conv1 (7x7)\nRF = 3x3', fontsize=12)
ax.invert_yaxis()
ax.set_xticks([])
ax.set_yticks([])

# Mark RF on input
axes[0].add_patch(mpatches.Rectangle((3-0.5, 3-0.5), 3, 3,
                                      facecolor='red', edgecolor='red',
                                      linewidth=2, alpha=0.3))
axes[0].text(4, 4, 'RF=3x3', ha='center', va='center', fontsize=8,
             fontweight='bold', color='red')

# Layer 2: another 3x3 conv -> output 5x5, RF=5x5
ax = axes[2]
out2_size = 5
ax.set_xlim(-0.5, out2_size - 0.5)
ax.set_ylim(-0.5, out2_size - 0.5)
ax.set_aspect('equal')
for i in range(out2_size):
    for j in range(out2_size):
        rect = mpatches.Rectangle((j-0.5, i-0.5), 1, 1,
                                  facecolor='#E8F5E9', edgecolor='gray', linewidth=0.5)
        ax.add_patch(rect)
rect = mpatches.Rectangle((2-0.5, 2-0.5), 1, 1,
                          facecolor='green', edgecolor='black', linewidth=2, alpha=0.7)
ax.add_patch(rect)
ax.set_title(f'After Conv2 (5x5)\nRF = 5x5', fontsize=12)
ax.invert_yaxis()
ax.set_xticks([])
ax.set_yticks([])

# Mark RF on input
axes[0].add_patch(mpatches.Rectangle((2-0.5, 2-0.5), 5, 5,
                                      facecolor='green', edgecolor='green',
                                      linewidth=2, alpha=0.2))
axes[0].text(4, 2, 'RF=5x5', ha='center', va='center', fontsize=8,
             fontweight='bold', color='green')

# Layer 3: another 3x3 conv -> output 3x3, RF=7x7
ax = axes[3]
out3_size = 3
ax.set_xlim(-0.5, out3_size - 0.5)
ax.set_ylim(-0.5, out3_size - 0.5)
ax.set_aspect('equal')
for i in range(out3_size):
    for j in range(out3_size):
        rect = mpatches.Rectangle((j-0.5, i-0.5), 1, 1,
                                  facecolor='#FFF3E0', edgecolor='gray', linewidth=0.5)
        ax.add_patch(rect)
rect = mpatches.Rectangle((1-0.5, 1-0.5), 1, 1,
                          facecolor='orange', edgecolor='black', linewidth=2, alpha=0.7)
ax.add_patch(rect)
ax.set_title(f'After Conv3 (3x3)\nRF = 7x7', fontsize=12)
ax.invert_yaxis()
ax.set_xticks([])
ax.set_yticks([])

# Mark RF on input
axes[0].add_patch(mpatches.Rectangle((1-0.5, 1-0.5), 7, 7,
                                      facecolor='orange', edgecolor='orange',
                                      linewidth=2, alpha=0.15))
axes[0].text(4, 1, 'RF=7x7', ha='center', va='center', fontsize=8,
             fontweight='bold', color='darkorange')

plt.suptitle('Receptive Field Grows with Network Depth', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Receptive field formula (stacking 3x3 convs without padding):')
print('  Layer 1: RF = 3x3 (local edges)')
print('  Layer 2: RF = 5x5 (textures, patterns)')
print('  Layer 3: RF = 7x7 (object parts)')
print('  Layer n: RF = (2n+1) x (2n+1)')
print('\nThis is why deeper layers learn more "global" / semantic features')

In [ ]:
# 感受野增長的定量視覺化
def compute_rf(n_layers, kernel_size=3, stride=1, padding=0):
    """Compute receptive field size after n conv layers."""
    rf = 1
    jump = 1
    for _ in range(n_layers):
        rf = rf + (kernel_size - 1) * jump
        jump = jump * stride
    return rf

layers = np.arange(1, 11)
rf_3x3 = [compute_rf(n, kernel_size=3) for n in layers]
rf_5x5 = [compute_rf(n, kernel_size=5) for n in layers]
rf_3x3_s2 = [compute_rf(n, kernel_size=3, stride=2) for n in layers]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(layers, rf_3x3, 'bo-', linewidth=2, markersize=8, label='3x3 kernel, stride=1')
ax.plot(layers, rf_5x5, 'rs-', linewidth=2, markersize=8, label='5x5 kernel, stride=1')
ax.plot(layers, rf_3x3_s2, 'g^-', linewidth=2, markersize=8, label='3x3 kernel, stride=2')

ax.set_xlabel('Number of Conv Layers', fontsize=12)
ax.set_ylabel('Receptive Field Size', fontsize=12)
ax.set_title('Receptive Field Growth with Network Depth', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(layers)

plt.tight_layout()
plt.show()

print('Key insight: Two 3x3 convs have the same RF (5x5) as one 5x5 conv,')
print('but use fewer parameters (2*9=18 vs 25) and have more non-linearity.')
print('This is why VGGNet uses only 3x3 convolutions.')

---
## Part 10: 練習題 Exercises

完成以下練習來鞏固本週所學。

### Exercise 1: Custom edge-detection kernel

實作一個自訂的邊緣偵測卷積核，並應用到 digits 資料集的 5 張不同數字上。

嘗試設計能偵測以下方向的卷積核：
- 45 度對角線
- 圓形邊緣（Laplacian of Gaussian 近似）

比較不同核的效果。

In [ ]:
# Exercise 1: Custom edge detection kernels
# TODO: Design custom kernels for 45-degree diagonal and circular edges
# TODO: Apply to 5 different digit images and display results

# diag_45_kernel = np.array([...])
# log_kernel = np.array([...])
#
# fig, axes = plt.subplots(5, 3, figsize=(12, 15))
# for row, d_idx in enumerate([0, 5, 15, 25, 35]):
#     img = images[d_idx].astype(float)
#     axes[row, 0].imshow(img, cmap='gray')
#     axes[row, 1].imshow(convolve2d(img, diag_45_kernel, mode='same'), cmap='gray')
#     axes[row, 2].imshow(convolve2d(img, log_kernel, mode='same'), cmap='gray')
# plt.show()

### Exercise 2: Compare max pooling vs average pooling on noisy images

1. 取 5 張 digit 影像，加入不同程度的高斯雜訊 (sigma = 0.5, 1.0, 2.0)
2. 分別做 Max Pooling 和 Average Pooling
3. 比較兩種池化對雜訊的抵抗能力

提示：用 `np.random.randn() * sigma` 加雜訊

In [ ]:
# Exercise 2: Pooling on noisy images
# TODO: Add Gaussian noise with different sigma values
# TODO: Apply max_pool2d and avg_pool2d
# TODO: Compare which pooling is more robust to noise

# img = images[0].astype(float)
# for sigma in [0.5, 1.0, 2.0]:
#     noisy_img = img + np.random.randn(*img.shape) * sigma
#     noisy_img = np.clip(noisy_img, 0, 16)
#     max_p = max_pool2d(noisy_img)
#     avg_p = avg_pool2d(noisy_img)
#     # Plot and compare ...

### Exercise 3: Design filters for horizontal/vertical/diagonal edges

設計一組 4 個 3x3 卷積核，分別偵測：
1. 水平邊緣
2. 垂直邊緣
3. 45 度對角線邊緣
4. 135 度對角線邊緣

將它們應用到 3-4 張 digit 影像上，顯示在一個 4x5 的圖表中（含原圖）。

In [ ]:
# Exercise 3: Design directional edge filters
# TODO: Create 4 kernels for H, V, 45-deg, 135-deg edges
# TODO: Apply to multiple digit images
# TODO: Display results in a grid

# k_horizontal = np.array([[-1,-1,-1],[0,0,0],[1,1,1]])
# k_vertical   = np.array([[-1,0,1],[-1,0,1],[-1,0,1]])
# k_diag45     = np.array([[0,-1,-1],[1,0,-1],[1,1,0]])
# k_diag135    = np.array([[-1,-1,0],[-1,0,1],[0,1,1]])
#
# fig, axes = plt.subplots(4, 5, figsize=(16, 12))
# for row, d_idx in enumerate([0, 1, 11, 7]):
#     img = images[d_idx].astype(float)
#     axes[row, 0].imshow(img, cmap='gray')
#     for col, (k_name, kernel) in enumerate([
#         ('H-edge', k_horizontal), ('V-edge', k_vertical),
#         ('45-deg', k_diag45), ('135-deg', k_diag135)]):
#         result = convolve2d(img, kernel, mode='same')
#         axes[row, col+1].imshow(np.abs(result), cmap='hot')
# plt.show()

---
## 本週實作總結 Lab Summary

在這次實作中，我們完成了：

1. **卷積運算**：從零實作 1D 和 2D 卷積，理解滑動窗口 + 加權求和
2. **常見卷積核**：Sobel、Laplacian、高斯模糊、銳化，觀察不同核的效果
3. **Padding & Stride**：視覺化不同填充和步幅對輸出尺寸的影響
4. **池化操作**：從零實作 Max Pooling 和 Average Pooling，比較差異
5. **CNN 架構圖**：視覺化典型 CNN 結構（Conv -> Pool -> FC）
6. **特徵圖視覺化**：多個濾波器產生的特徵圖呈現不同空間模式
7. **CNN vs MLP**：展示卷積特徵提取相比扁平像素的優勢
8. **CAM 概念**：簡化版 CAM 演示，理解模型如何聚焦特定區域
9. **感受野**：視覺化感受野隨深度增長，理解深層特徵的「全局性」

### 關鍵收穫 Key Takeaways
- 卷積核是 CNN 的核心，它們在訓練中自動學習最佳的特徵偵測器
- 池化降低空間維度，同時增加平移不變性
- 淺層學邊緣，深層學語義 -- 層次化特徵學習是深度學習的本質
- CAM/Grad-CAM 是理解 CNN 決策的重要可解釋性工具
- 兩個 3x3 conv 等效於一個 5x5 conv 的感受野，但更高效

### 下週預告 Next Week Preview
**第 13 週：RNN 與 Transformer** -- 序列模型、注意力機制與自然語言處理視覺化